# Test split — the leakage audit and the fourteen rows

Steps 3, 4 and 5 of `docs/preregistration_test.md`: the leakage audit, the twelve local rows,
and the two commercial rows. Step 2, the spend authorization, is a dated line in `docs/budget.md`
and is checked in section 10 before anything is bought.

This is the run that opens the seal. Sections 1-9 spend nothing and run with no API key in the
kernel; section 10 is the only paid part.

---
## 1 — Host and working tree

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv

In [ ]:
from pathlib import Path

if not Path('manage.py').exists():
    if not Path('Style-Aware-MT/manage.py').exists():
        !git clone https://github.com/prnamhr/Style-Aware-MT.git
    %cd Style-Aware-MT
!git pull --ff-only
!git rev-parse --short HEAD

In [ ]:
!git log --oneline -1 -- docs/preregistration_test.md

In [ ]:
# %pip installs into the kernel; !pip may not.
%pip install -r requirements.txt

In [ ]:
# torch 2.12 breaks the pinned-torch ABI these three ship against; the pipeline is text-only.
%pip uninstall -q -y torchvision torchaudio torchcodec

In [ ]:
import torch

cap = torch.cuda.get_device_capability(0)
print(f'{torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}',
      f'torch {torch.__version__} / cuda {torch.version.cuda}')
assert torch.cuda.is_bf16_supported(), 'bf16 unsupported; the frozen base is not quantized'

---
## 2 — Run parameters

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import time
from datetime import datetime, timedelta, timezone

import yaml

SPLIT = 'test'
EVAL_FILE = Path('data/splits/test.jsonl')

# (condition, committed config). The three RLSF configs carry their own output.name.
ROWS = [
    ('zeroshot',       'configs/base_qwen.yaml'),
    ('random_fewshot', 'configs/base_qwen.yaml'),
    ('knn_fewshot',    'configs/base_qwen.yaml'),
    ('sparse_knn',     'configs/sparse_knn.yaml'),
    ('afsp_margin',    'configs/base_qwen.yaml'),
    ('afsp_full',      'configs/base_qwen.yaml'),
    ('peft',           'configs/peft_qwen.yaml'),
    ('peft_knn',       'configs/peft_afsp.yaml'),
    ('peft_afsp',      'configs/peft_afsp.yaml'),
    ('peft',           'configs/rlsf_eval_w3_0.0.yaml'),
    ('peft',           'configs/rlsf_eval_w3_2.0.yaml'),
    ('peft',           'configs/rlsf_eval_w3_6.0.yaml'),
]
assert len(ROWS) == 12

In [ ]:
HASHES = json.loads(Path('data/splits/hashes.json').read_text(encoding='utf-8'))
TEST_SHA = hashlib.sha256(EVAL_FILE.read_bytes()).hexdigest()
assert TEST_SHA == HASHES['hashes']['test.jsonl'], 'test.jsonl is not the committed split'

SEGMENTS = [json.loads(x) for x in EVAL_FILE.open(encoding='utf-8') if x.strip()]
TEST_SRC = [r['input'] for r in SEGMENTS]
assert len(SEGMENTS) == HASHES['counts']['final']['test'] == 1322, len(SEGMENTS)
print(f'{len(SEGMENTS)} segments  {TEST_SHA[:16]}')

In [ ]:
# Every frozen value the pre-registration's settings table names, read back from the configs.
CFGS = {p: yaml.safe_load(Path(p).read_text(encoding='utf-8')) for _, p in ROWS}

for path, cfg in CFGS.items():
    gen = cfg['generator']
    assert gen['model'] == 'Qwen/Qwen2.5-7B-Instruct', (path, gen['model'])
    assert (gen['temperature'], gen['top_p']) == (0.0, 1.0), f'{path}: not the locked greedy decoding'
    assert (gen['max_tokens'], gen['seed']) == (1024, 42), (path, gen)
    assert gen['dtype'] == 'bfloat16' and gen['load_in_4bit'] is False, f'{path}: base redefined'

BASE, SPARSE = CFGS['configs/base_qwen.yaml'], CFGS['configs/sparse_knn.yaml']
assert BASE['retrieval']['k'] == 8 and BASE['retrieval']['index_dir'] == 'data/knn_index'
assert (BASE['afsp']['beta'], BASE['afsp']['lambda_style']) == (0.3, 0.75), BASE['afsp']
assert BASE['afsp']['style_target_sigma'] == 1.0 and BASE['afsp']['style_objective'] == 'bandpass'
assert (SPARSE['rarity']['min_df'], SPARSE['rarity']['freeze_n']) == (40, 500), SPARSE['rarity']
assert SPARSE['sparse']['m'] == 4, SPARSE['sparse']

ADAPTERS = {p: c['generator']['adapter_path'] for p, c in CFGS.items() if 'adapter_path' in c['generator']}
assert set(ADAPTERS.values()) == {
    'models/peft_lora_r32_lr2e-4/checkpoint-1358',
    'models/rlsf_grpo_w3_0.0/checkpoint-200',
    'models/rlsf_grpo_w3_2.0/checkpoint-200',
    'models/rlsf_grpo_w3_6.0/checkpoint-100',
}, ADAPTERS
print('\n'.join(f'{p:<34} {a}' for p, a in ADAPTERS.items()))

In [ ]:
import getpass
import logging

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')
# The local pass runs key-free, so it cannot spend. Section 10 sets the keys; if you have
# already been there, restart the kernel before re-running this cell.
for var in ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY'):
    assert not os.environ.get(var), f'{var} is set; sections 1-9 make no paid call'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
logging.getLogger('httpx').setLevel(logging.WARNING)

BUDGET_H = 12
DEADLINE = datetime.now(timezone.utc) + timedelta(hours=BUDGET_H)
print(f'HF_TOKEN set, no rater key present. Deadline {DEADLINE:%Y-%m-%d %H:%M}Z')

---
## 3 — Opening the seal

The committed configs point at `val.jsonl` and stay that way. Each row runs from a derived
config written under `configs/test/`, identical to its source except for `data.eval_file`.
The derived files are recorded in the manifest by digest, so which bytes generated the test
rows is answerable afterwards.

`SEAL_OPEN` gates every cell below that reads `data/splits/test.jsonl`. Set it by hand.

In [ ]:
SEAL_OPEN = False

In [ ]:
assert SEAL_OPEN, 'set SEAL_OPEN = True to read the sealed split'

TEST_CFG_DIR = Path('configs/test')
TEST_CFG_DIR.mkdir(parents=True, exist_ok=True)
DERIVED = {}

for path in sorted(CFGS):
    cfg = json.loads(json.dumps(CFGS[path]))  # deep copy; the loaded dicts are reused below
    assert cfg['data']['eval_file'] == 'data/splits/val.jsonl', (path, cfg['data'])
    cfg['data']['eval_file'] = str(EVAL_FILE)
    dst = TEST_CFG_DIR / Path(path).name
    dst.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
    DERIVED[path] = {'path': str(dst), 'sha256': hashlib.sha256(dst.read_bytes()).hexdigest()}
    print(f'{path:<34} -> {dst}  {DERIVED[path]["sha256"][:12]}')

assert not subprocess.run(['git', 'diff', '--quiet', '--', 'configs'], check=False).returncode, \
    'a committed config was modified; the seal must be opened by derivation, not by editing'

---
## 4 — Weights and the index

In [ ]:
from huggingface_hub import snapshot_download

HF_REPO = 'prnamhr/style-aware-mt-models'
ADAPTER_FILES = ('adapter_config.json', 'adapter_model.safetensors')
ADAPTER_SHA = {}

for adapter in sorted(set(ADAPTERS.values())):
    a = Path(adapter)
    if not all((a / f).exists() for f in ADAPTER_FILES):
        snapshot_download(HF_REPO, local_dir='.', token=os.environ['HF_TOKEN'],
                          allow_patterns=[f'{adapter}/{f}' for f in ADAPTER_FILES])
    conf = json.loads((a / 'adapter_config.json').read_text(encoding='utf-8'))
    ADAPTER_SHA[adapter] = hashlib.sha256((a / 'adapter_model.safetensors').read_bytes()).hexdigest()
    print(f'{adapter:<44} r={conf["r"]} alpha={conf["lora_alpha"]}  {ADAPTER_SHA[adapter][:12]}')

In [ ]:
t0 = time.perf_counter()
snapshot_download('Qwen/Qwen2.5-7B-Instruct', token=os.environ['HF_TOKEN'], max_workers=8,
                  allow_patterns=['*.json', '*.safetensors', '*.txt', '*.jinja'])
print(f'base cached in {(time.perf_counter() - t0) / 60:.1f} min')

In [ ]:
INDEX = Path('data/knn_index')
INDEX_FILES = ('embeddings.npy', 'pairs.jsonl', 'meta.json')
INDEX_REBUILT = not all((INDEX / f).exists() for f in INDEX_FILES)

if INDEX_REBUILT:
    !python3 manage.py build_index --config configs/base_qwen.yaml

INDEX_SHA = {f: hashlib.sha256((INDEX / f).read_bytes()).hexdigest() for f in INDEX_FILES}
meta = json.loads((INDEX / 'meta.json').read_text(encoding='utf-8'))
assert meta['embed_model'] == BASE['retrieval']['embed_model'], meta
assert meta['indexed_side'] == 'source' and meta['n_passages'] == 10860, meta

# The unquarantined pool, the one every val row retrieved from. Section 5 audits it, and
# under the declared trigger does not replace it.
print(f'index {"rebuilt" if INDEX_REBUILT else "transferred"}, {meta["n_passages"]} passages')
for f, digest in INDEX_SHA.items():
    print(f'  {f:16s} {digest[:12]}')

---
## 5 — Step 3: the leakage audit

Diagnostic. Without `--write-quarantine`, which would overwrite the val-only 22-row list at
`data/splits/pool_quarantine.json` with one conditioned on the sealed split.

In [ ]:
assert SEAL_OPEN
QUARANTINE = Path('data/splits/pool_quarantine.json')
QUARANTINE_SHA = hashlib.sha256(QUARANTINE.read_bytes()).hexdigest()

!python3 manage.py leakage --config configs/sparse_retrieval.yaml --split test --unseal-test

assert hashlib.sha256(QUARANTINE.read_bytes()).hexdigest() == QUARANTINE_SHA, \
    'the val-only quarantine was overwritten'

In [ ]:
LEAK = json.loads(Path('results/leakage_test.json').read_text(encoding='utf-8'))
RATE = LEAK['n_eval_rows_flagged'] / LEAK['n_eval_rows']
TRIGGER = 0.0256  # twice the val rate of 17/1323, declared in docs/preregistration_test.md

print(f'{LEAK["n_eval_rows_flagged"]}/{LEAK["n_eval_rows"]} test segments flagged = {RATE:.2%}')
print(f'{LEAK["n_pool_rows_flagged"]} pool rows flagged, {LEAK["n_flags"]} flags')
print(json.dumps(LEAK['max_cos_histogram'], indent=2))
print(f'\ntrigger {TRIGGER:.2%}: ' + ('FIRED — the sensitivity rerun is owed' if RATE > TRIGGER
      else 'not fired — generation proceeds on the unquarantined pool'))

---
## 6 — The gate

In [ ]:
from src.infer.run import (_load_configured_glossary, build_fewshot_user, make_client,
                           order_exemplars, resolve_out_name)
from src.retrieval.retrieve import RetrievalIndex

index = RetrievalIndex(BASE['retrieval']['index_dir'], embed_model=BASE['retrieval']['embed_model'])
STYLE = Path(BASE['prompt']['style_instruction_file']).read_text(encoding='utf-8')
GLOSSARY = _load_configured_glossary(BASE)

PROBE_N = 8
probe_src = TEST_SRC[:PROBE_N]
probe = [build_fewshot_user(s, order_exemplars(ex, BASE['prompt']['ordering']), GLOSSARY)
         for s, ex in zip(probe_src, index.retrieve(probe_src, k=BASE['retrieval']['k']))]

t0 = time.perf_counter()
client = make_client(BASE['generator'])
LOAD_S = time.perf_counter() - t0

t0 = time.perf_counter()
for user in probe:
    client.complete(STYLE, user)
SEG_S = (time.perf_counter() - t0) / PROBE_N

print(f'{LOAD_S:.0f}s base load, {SEG_S:.2f}s per segment at k=8')
print(f'{torch.cuda.max_memory_reserved() / 2**30:.1f} GiB reserved')

In [ ]:
# One process per row, so the base is loaded twelve times.
pass_h = (len(SEGMENTS) * SEG_S + LOAD_S) / 3600
left_h = (DEADLINE - datetime.now(timezone.utc)).total_seconds() / 3600
print(f'{pass_h:.2f} h per row, {pass_h * len(ROWS):.1f} h for {len(ROWS)}, {left_h:.1f} h left')

if pass_h * len(ROWS) > 0.9 * left_h:
    print('\nDoes not fit. Generation resumes per row, so a partial session is recoverable.')
else:
    print('\nFits. Section 7 may start.')

In [ ]:
del client, index
torch.cuda.empty_cache()

---
## 7 — Step 4: generation

In [ ]:
assert SEAL_OPEN
TIMING = {}

for cond, src_cfg in ROWS:
    cfg_path = DERIVED[src_cfg]['path']
    name = resolve_out_name(cond, CFGS[src_cfg])
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, 'manage.py', 'infer', '--condition', cond,
                        '--config', cfg_path], check=False)
    assert r.returncode == 0, f'{name} exited {r.returncode}'
    TIMING[name] = {'condition': cond, 'config': cfg_path,
                    'seconds': round(time.perf_counter() - t0, 1),
                    'finished': datetime.now(timezone.utc).isoformat()}
    print(f'{name}: {TIMING[name]["seconds"] / 60:.1f} min')

In [ ]:
NAMES = list(TIMING)
assert len(NAMES) == 12 and len(set(NAMES)) == 12, NAMES

OUTPUT_SHA = {}
for name in NAMES:
    path = Path(f'outputs/{name}_{SPLIT}.jsonl')
    rows = [json.loads(x) for x in path.open(encoding='utf-8') if x.strip()]
    assert len(rows) == len(SEGMENTS), f'{name}: {len(rows)} rows, expected {len(SEGMENTS)}'
    assert [r['input'] for r in rows] == TEST_SRC, f'{name}: source order differs from test.jsonl'
    blank = [i for i, r in enumerate(rows) if not r['prediction'].strip()]
    OUTPUT_SHA[name] = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f'{name:<16} {len(rows)} rows, {len(blank)} blank {blank[:5]}  {OUTPUT_SHA[name][:12]}')

---
## 8 — Manifest

`output_sha256` is the same digest the raters and COMET will bind their scores to.

In [ ]:
import platform

import peft as peft_lib
import transformers

MANIFEST = {
    'split': SPLIT,
    'eval_file': {'path': str(EVAL_FILE), 'sha256': TEST_SHA, 'n': len(SEGMENTS)},
    'commit': subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True,
                             text=True).stdout.strip(),
    'preregistration': 'docs/preregistration_test.md',
    'rows': {name: {**TIMING[name], 'output_sha256': OUTPUT_SHA[name]} for name in NAMES},
    'derived_configs': DERIVED,
    'adapters': ADAPTER_SHA,
    'index': {'dir': str(INDEX), 'rebuilt_here': INDEX_REBUILT, 'sha256': INDEX_SHA, 'meta': meta},
    'quarantine': {'path': str(QUARANTINE), 'sha256': QUARANTINE_SHA, 'applied': False},
    'leakage': {'flagged': LEAK['n_eval_rows_flagged'], 'n': LEAK['n_eval_rows'],
                'rate': round(RATE, 5), 'trigger': TRIGGER, 'fired': bool(RATE > TRIGGER)},
    'versions': {
        'device': torch.cuda.get_device_name(0),
        'torch': torch.__version__,
        'cuda': torch.version.cuda,
        'transformers': transformers.__version__,
        'peft': peft_lib.__version__,
        'python': platform.python_version(),
    },
}
MANIFEST_PATH = Path('outputs/test_manifest.json')
MANIFEST_PATH.write_text(json.dumps(MANIFEST, indent=2) + '\n', encoding='utf-8')
print(json.dumps(MANIFEST, indent=2))

---
## 9 — Seal on the local pass

In [ ]:
# 1. Nothing was spent.
for name in NAMES:
    usage = json.loads(Path(f'outputs/{name}_{SPLIT}_usage.json').read_text(encoding='utf-8'))
    assert usage.get('cost_usd', 0.0) == 0.0, (name, usage)
    print(f'{name:<16} {usage["calls"]} calls, ${usage.get("cost_usd", 0.0):.2f}')

# 2. No rater key was ever present.
for var in ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY'):
    assert not os.environ.get(var), var

# 3. The split is what it was, and nothing outside the test artifacts moved.
assert hashlib.sha256(EVAL_FILE.read_bytes()).hexdigest() == TEST_SHA, 'test.jsonl changed'
dirty = subprocess.run(['git', 'status', '--porcelain', 'configs', 'outputs', 'results', 'data'],
                       capture_output=True, text=True).stdout.splitlines()
unexpected = [l for l in dirty if 'test' not in l]
assert not unexpected, unexpected

print(f'\nsealed: 0 paid calls, {len(NAMES)} local test rows, no val artifact touched')

---
## 10 — Step 5: the two commercial rows

The only paid part. `PAID` names them, `SPEND_OK` gates them, and `AUTHORIZED_USD` has to
cover the projection before either runs.

Each row runs twice: a 20-segment pilot whose realized per-call rate is checked against the
projection, then the full pass, which resumes from the pilot's rows. The pilot's usage sidecar
is copied aside first, because `manage.py infer` overwrites it per invocation and the blended
rate is otherwise unrecoverable.

In [ ]:
PAID = [
    ('zeroshot',   'configs/commercial_haiku_zeroshot.yaml',      4.774e-4),
    ('sparse_knn', 'configs/commercial_gpt56_sparse_knn.yaml',    7.774e-3),
]
PILOT_N = 20

PAID_CFGS = {p: yaml.safe_load(Path(p).read_text(encoding='utf-8')) for _, p, _ in PAID}
HAIKU = PAID_CFGS['configs/commercial_haiku_zeroshot.yaml']
GPT = PAID_CFGS['configs/commercial_gpt56_sparse_knn.yaml']

assert HAIKU['generator']['model'] == 'claude-haiku-4-5', HAIKU['generator']
assert HAIKU['generator']['thinking'] is False and HAIKU['generator']['temperature'] == 0.0
assert GPT['generator']['model'] == 'gpt-5.6-sol', GPT['generator']
assert GPT['generator']['reasoning_effort'] == 'none', 'thinking would change what the row measures'
assert GPT['retrieval']['k'] == 8 and GPT['retrieval']['index_dir'] == 'data/knn_index'
assert (GPT['rarity']['min_df'], GPT['rarity']['freeze_n']) == (40, 500), GPT['rarity']
assert GPT['sparse']['m'] == 4, GPT['sparse']

PROJECTED = {resolve_out_name(c, PAID_CFGS[p]): rate * len(SEGMENTS) for c, p, rate in PAID}
PROJECTED_TOTAL = sum(PROJECTED.values())
for name, usd in PROJECTED.items():
    print(f'{name:<20} {len(SEGMENTS)} calls  ${usd:.2f}')
print(f'{"total":<20} {2 * len(SEGMENTS)} calls  ${PROJECTED_TOTAL:.2f}')

In [ ]:
SPEND_OK = False
AUTHORIZED_USD = 0.0

In [ ]:
assert SEAL_OPEN and SPEND_OK, 'set SPEND_OK = True to buy the two commercial rows'
assert AUTHORIZED_USD >= PROJECTED_TOTAL, (
    f'${AUTHORIZED_USD:.2f} authorized against a ${PROJECTED_TOTAL:.2f} projection'
)
print(subprocess.run(['git', 'log', '-1', '--format=%h %ad %s', '--date=short', '--',
                      'docs/budget.md'], capture_output=True, text=True).stdout)
print('the authorization this run spends against must already be a dated line in docs/budget.md')

In [ ]:
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
%pip install -q anthropic==0.109.1 openai==2.41.1

In [ ]:
# Two derived configs per row: the pilot caps data.limit, the full pass lifts it.
PAID_DERIVED = {}
for cond, src, _ in PAID:
    for tag, limit in (('pilot', PILOT_N), ('full', None)):
        cfg = json.loads(json.dumps(PAID_CFGS[src]))
        assert cfg['data']['eval_file'] == 'data/splits/val.jsonl', src
        cfg['data']['eval_file'] = str(EVAL_FILE)
        cfg['data']['limit'] = limit
        dst = TEST_CFG_DIR / f'{Path(src).stem}.{tag}.yaml'
        dst.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
        PAID_DERIVED[(src, tag)] = str(dst)
        print(f'{dst}  limit={limit}')

In [ ]:
import shutil

PILOT = {}
for cond, src, rate in PAID:
    name = resolve_out_name(cond, PAID_CFGS[src])
    r = subprocess.run([sys.executable, 'manage.py', 'infer', '--condition', cond,
                        '--config', PAID_DERIVED[(src, 'pilot')]], check=False)
    assert r.returncode == 0, f'{name} pilot exited {r.returncode}'

    sidecar = Path(f'outputs/{name}_{SPLIT}_usage.json')
    shutil.copy(sidecar, sidecar.with_name(f'{name}_{SPLIT}_pilot_usage.json'))
    u = json.loads(sidecar.read_text(encoding='utf-8'))
    # A zero here is an unpriced model, not a free one, and it would make the guard below inert.
    assert u['cost_usd'] > 0, f'{name}: cost_usd is 0; the model has no pricing entry'
    realized = u['cost_usd'] / u['calls']
    PILOT[name] = {'calls': u['calls'], 'cost_usd': u['cost_usd'], 'rate': realized}
    print(f'{name:<20} {u["calls"]} calls  ${u["cost_usd"]:.4f}  ${realized:.3e}/call  '
          f'projected ${rate:.3e}  x{realized / rate:.2f}')

In [ ]:
# A rate this far above the projection means the pilot is not what was priced.
for cond, src, rate in PAID:
    name = resolve_out_name(cond, PAID_CFGS[src])
    assert PILOT[name]['rate'] <= 1.25 * rate, (
        f'{name}: ${PILOT[name]["rate"]:.3e}/call is more than 1.25x the projected ${rate:.3e}; '
        f'the full pass would cost about ${PILOT[name]["rate"] * len(SEGMENTS):.2f}'
    )
revised = sum(PILOT[resolve_out_name(c, PAID_CFGS[p])]['rate'] * len(SEGMENTS) for c, p, _ in PAID)
print(f'at the realized rates the full pass is ${revised:.2f} against ${PROJECTED_TOTAL:.2f} '
      f'projected and ${AUTHORIZED_USD:.2f} authorized')
assert revised <= AUTHORIZED_USD, 'the realized rates exceed the authorization'

In [ ]:
for cond, src, _ in PAID:
    name = resolve_out_name(cond, PAID_CFGS[src])
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, 'manage.py', 'infer', '--condition', cond,
                        '--config', PAID_DERIVED[(src, 'full')]], check=False)
    assert r.returncode == 0, f'{name} exited {r.returncode}'
    TIMING[name] = {'condition': cond, 'config': PAID_DERIVED[(src, 'full')],
                    'seconds': round(time.perf_counter() - t0, 1),
                    'finished': datetime.now(timezone.utc).isoformat()}
    print(f'{name}: {TIMING[name]["seconds"] / 60:.1f} min')

In [ ]:
SPEND = {}
for cond, src, _ in PAID:
    name = resolve_out_name(cond, PAID_CFGS[src])
    path = Path(f'outputs/{name}_{SPLIT}.jsonl')
    rows = [json.loads(x) for x in path.open(encoding='utf-8') if x.strip()]
    assert len(rows) == len(SEGMENTS), f'{name}: {len(rows)} rows, expected {len(SEGMENTS)}'
    assert [r['input'] for r in rows] == TEST_SRC, f'{name}: source order differs from test.jsonl'
    blank = [i for i, r in enumerate(rows) if not r['prediction'].strip()]
    OUTPUT_SHA[name] = hashlib.sha256(path.read_bytes()).hexdigest()

    full = json.loads(Path(f'outputs/{name}_{SPLIT}_usage.json').read_text(encoding='utf-8'))
    SPEND[name] = {'pilot': PILOT[name], 'full': {k: full[k] for k in ('calls', 'cost_usd')},
                   'total_calls': PILOT[name]['calls'] + full['calls'],
                   'total_usd': round(PILOT[name]['cost_usd'] + full['cost_usd'], 4)}
    print(f'{name:<20} {len(rows)} rows, {len(blank)} blank  '
          f'${SPEND[name]["total_usd"]:.4f} over {SPEND[name]["total_calls"]} calls  '
          f'{OUTPUT_SHA[name][:12]}')

TOTAL_USD = round(sum(v['total_usd'] for v in SPEND.values()), 4)
print(f'\npaid this session: ${TOTAL_USD:.4f} against ${AUTHORIZED_USD:.2f} authorized')

---
## 11 — Final seal

In [ ]:
ALL = list(TIMING)
assert len(ALL) == 14 and len(set(ALL)) == 14, ALL

for name in ALL:
    path = Path(f'outputs/{name}_{SPLIT}.jsonl')
    assert path.exists() and OUTPUT_SHA[name] == hashlib.sha256(path.read_bytes()).hexdigest(), name

# The twelve local rows are still free; only the two commercial rows cost anything.
for name in NAMES:
    usage = json.loads(Path(f'outputs/{name}_{SPLIT}_usage.json').read_text(encoding='utf-8'))
    assert usage.get('cost_usd', 0.0) == 0.0, (name, usage)

assert hashlib.sha256(EVAL_FILE.read_bytes()).hexdigest() == TEST_SHA, 'test.jsonl changed'
assert hashlib.sha256(QUARANTINE.read_bytes()).hexdigest() == QUARANTINE_SHA, 'quarantine changed'
dirty = subprocess.run(['git', 'status', '--porcelain', 'configs', 'outputs', 'results', 'data'],
                       capture_output=True, text=True).stdout.splitlines()
unexpected = [l for l in dirty if 'test' not in l]
assert not unexpected, unexpected
print(f'{len(ALL)} rows on test, ${TOTAL_USD:.4f} spent, no val artifact touched')

In [ ]:
MANIFEST['rows'] = {name: {**TIMING[name], 'output_sha256': OUTPUT_SHA[name]} for name in ALL}
MANIFEST['derived_configs'] = {**DERIVED,
                               **{f'{k[0]}::{k[1]}': v for k, v in PAID_DERIVED.items()}}
MANIFEST['spend'] = {'authorized_usd': AUTHORIZED_USD, 'projected_usd': round(PROJECTED_TOTAL, 4),
                     'actual_usd': TOTAL_USD, 'per_row': SPEND}
MANIFEST_PATH.write_text(json.dumps(MANIFEST, indent=2) + '\n', encoding='utf-8')
print(json.dumps(MANIFEST['spend'], indent=2))

In [ ]:
!tar -czf test_generation.tar.gz outputs/*_test.jsonl outputs/*_test_usage.json \
    outputs/*_test_pilot_usage.json outputs/test_manifest.json results/leakage_test.json configs/test
!ls -la test_generation.tar.gz
!git status --short outputs results configs